# siann quickstart

siann is fast similarity search on Apple silicon for dense vectors. Give it base vectors and a metric, and
it finds each query's nearest neighbors, on the GPU (through MLX) or the CPU (BLAS through NumPy, and
siann's own Rust kernels), choosing the device itself unless you tell it which. This notebook covers exact
search: every metric, the scores it returns, how it explains its choices, and how the devices compare.

In [1]:
import time

import numpy as np
import siann

siann.__version__

'0.1.0'

## Some vectors

Real embeddings cluster, so instead of uniform noise the base vectors here are 200,000 points in 128
dimensions drawn around 1,000 random centers, and the queries are 1,000 more points drawn the same way.

In [2]:
rng = np.random.default_rng(0)
centers = rng.standard_normal((1000, 128), dtype=np.float32)


def sample(n):
    return (centers[rng.integers(0, len(centers), n)]
            + 0.3 * rng.standard_normal((n, 128), dtype=np.float32))


xb, xq = sample(200_000), sample(1000)
xb.shape, xq.shape

((200000, 128), (1000, 128))

## Exact search

`siann.Index(xb, metric)` builds an index, and `index.search(xq, k)` returns each query's `k` nearest base
vectors, best first: `indices` (int64) and `scores`. Without a recall target the search is exact, so these
are the true nearest neighbors.

In [3]:
index = siann.Index(xb, metric="cosine")
res = index.search(xq, k=5)
res.indices[:3], res.scores[:3]

(array([[ 42308, 129391,  25955,  14467,  39531],
        [ 35068, 176086,  29961, 145560, 174455],
        [183980, 132183,  61172, 139163, 132091]]),
 array([[0.9333738 , 0.9273064 , 0.9251599 , 0.92499685, 0.9248781 ],
        [0.95301634, 0.9448018 , 0.9411995 , 0.9390446 , 0.9351648 ],
        [0.946916  , 0.94622344, 0.94547373, 0.94499993, 0.94487053]],
       dtype=float32))

`index.plan` says what was chosen and why: here, which device runs the search and how each device would
compute it.

In [4]:
print(index.plan)

siann.Index: 200,000 vectors, 128 dimensions, metric cosine
  kind: exact
  device: gpu
  exact search (no recall target): the results are the true nearest neighbors
  GPU found: exact search on the GPU, measured faster than the CPU in batches (1,000 queries, every metric; 100, L2 and inner product)
  on the CPU: a BLAS matrix multiply through NumPy and siann's Rust top-k
  cosine: vectors normalized to unit length and ranked by inner product


## The metrics and their scores

siann searches by L2 (Euclidean), L1, L-infinity (the largest coordinate difference), cosine and inner
product. Scores are each metric's natural value: the distance for L2 (not squared), L1 and L-infinity,
smallest first, and the similarity for cosine and inner product, largest first.

In [5]:
for metric in siann.METRICS:
    r = siann.Index(xb, metric).search(xq[:1], k=3)
    print(f"{metric:7s} neighbors {r.indices[0]}  scores {np.round(r.scores[0], 3)}")

l2      neighbors [42308 14467 39531]  scores [3.907 4.039 4.054]
l1      neighbors [42308 39531 92202]  scores [34.244 36.218 36.469]
linf    neighbors [144869  91583 135196]  scores [0.81  0.814 0.82 ]
cosine  neighbors [ 42308 129391  25955]  scores [0.933 0.927 0.925]
ip      neighbors [129391 127843  92498]  scores [107.948 107.343 106.861]


The scores are exact: here they are recomputed with NumPy for the L2 search.

In [6]:
r = siann.Index(xb, "l2").search(xq[:4], k=3)
np.allclose(r.scores, np.linalg.norm(xb[r.indices] - xq[:4, None, :], axis=2), rtol=1e-4)

True

## Choosing the device

`device="cpu"` or `device="gpu"` pins an index to one device, and the same argument to `search` overrides it
for one call. Each device makes its own copy of the vectors the first time a search needs it. Both return the
same neighbors; how fast each is depends on the metric. L2, cosine and inner product are matrix multiplies
on either device, where L1 and L-infinity have no matrix-multiply form, so the GPU's lead is largest there.
The times below are for this machine. Each is the second of two identical calls: the first builds the
device's copy of the vectors and, on the GPU, compiles the kernels for this batch size and k.

In [7]:
def timed(index, device):
    index.search(xq, 10, device=device)                   # builds the copy, compiles for this shape
    start = time.perf_counter()
    index.search(xq, 10, device=device)
    return time.perf_counter() - start


devices = ["cpu", "gpu"] if index.plan.device == "gpu" else ["cpu"]
for metric in siann.METRICS:
    idx = siann.Index(xb, metric)
    times = {d: timed(idx, d) for d in devices}
    print(f"{metric:7s} " + "  ".join(f"{d} {t * 1e3:7.1f} ms" for d, t in times.items()))

l2      cpu    49.4 ms  gpu    14.2 ms


l1      cpu   112.4 ms  gpu    22.0 ms


linf    cpu   138.3 ms  gpu    22.5 ms
cosine  cpu    50.2 ms  gpu    14.4 ms


ip      cpu    50.4 ms  gpu    15.6 ms


## When fewer than k exist

Asking for more neighbors than there are vectors returns all of them, then `-1` in the missing slots, with
the metric's worst score (infinity for a distance, minus infinity for a similarity). A neighbor is never
made up.

In [8]:
small = siann.Index(xb[:3], "l2")
small.search(xq[:1], k=5)

SearchResult(indices=array([[ 2,  0,  1, -1, -1]]), scores=array([[15.428913, 15.798179, 15.825872,       inf,       inf]],
      dtype=float32))

## What siann refuses

siann searches dense vectors, so a sparse matrix is refused with an explanation rather than converted
silently; so is a recall target, which asks for approximate search.

In [9]:
import scipy.sparse

for attempt in (lambda: siann.Index(scipy.sparse.random(100, 20, density=0.1, format="csr"), "ip"),
                lambda: siann.Index(xb, "l2", recall=0.95)):
    try:
        attempt()
    except NotImplementedError as e:
        print("NotImplementedError:", e)

NotImplementedError: xb is a sparse matrix: siann searches dense vectors; pass a NumPy array (x.toarray() for a small one)
NotImplementedError: recall targets (approximate search) are not supported in this version: use recall=None for exact search
